In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# ============================================================
# EDA FUNCTIONS
# ============================================================

def data_quality(df):
    print("\n[2] DATA QUALITY CHECK")
    print("-" * 40)
    print("\nMissing values:")
    print(df.isna().sum())
    print("\nMissing percentage:")
    print((df.isna().mean() * 100).round(2))
    print(f"\nDuplicate rows: {df.duplicated().sum()}")
    return df

def check_missing_gaps(df, target_col):
    print("\n[2.5] MISSING GAPS ANALYSIS")
    print("-" * 40)
    if target_col in df.columns:
        missing_mask = df[target_col].isna()
        gap_starts = missing_mask & ~missing_mask.shift(1).fillna(False)
        gap_ends = missing_mask & ~missing_mask.shift(-1).fillna(False)

        starts_idx = df.index[gap_starts]
        ends_idx = df.index[gap_ends]

        if len(starts_idx) > 0 and len(ends_idx) > 0:
            df['_is_missing'] = df[target_col].isna().astype(int)
            df['_block'] = (df['_is_missing'].diff() != 0).cumsum()
            gaps = df[df['_is_missing'] == 1].groupby('_block').size()

            print(f"Total missing hours: {df[target_col].isna().sum()} in {len(gaps)} gaps")
            df.drop(columns=['_is_missing', '_block'], inplace=True)
        else:
            print("No missing value gaps found in the target column.")

def check_outliers(df, target_col):
    print("\n[3.5] OUTLIER CHECK (3-Sigma)")
    print("-" * 40)
    if target_col in df.columns:
        mu = df[target_col].mean()
        std = df[target_col].std()
        lower = mu - 3 * std
        upper = mu + 3 * std
        outliers = df[(df[target_col] < lower) | (df[target_col] > upper)]
        print(f"Mean = {mu:,.0f} MW, Std = {std:,.0f} MW -> {len(outliers)} hours beyond +/- 3 std devs")

def descriptive_stats(df):
    print("\n[3] DESCRIPTIVE STATISTICS")
    print("-" * 40)
    display(df.describe())

# ============================================================
# FEATURE ENGINEERING & MODELING PIPELINE
# ============================================================

def run_comprehensive_pipeline(file_path):
    print("=" * 80)
    print("   COMPREHENSIVE ENERGY DEMAND FORECASTING WORKFLOW")
    print("=" * 80)

    print("\n--- 1. LOAD DATA & INITIAL PREP ---")
    df = pd.read_csv(file_path)

    # Handle flexible datetime column names
    date_col = next((c for c in ['Datetime', 'timestamp', 'datetime'] if c in df.columns), None)
    if date_col:
        df[date_col] = pd.to_datetime(df[date_col], format='mixed', dayfirst=True)
        df.sort_values(by=date_col, inplace=True)
        df.set_index(date_col, inplace=True)

    # Handle flexible target column names
    target_col = next((c for c in ['load_MW', 'MW'] if c in df.columns), None)
    if not target_col:
        raise ValueError("Could not find a target power load column (e.g. 'load_MW' or 'MW') in dataset.")

    print("\n=================== RUNNING EDA ===================")
    data_quality(df)
    check_missing_gaps(df, target_col)
    check_outliers(df, target_col)
    descriptive_stats(df)

    print("\n=================== FEATURE ENGINEERING ===================")
    # Extract cyclical datetime components
    df['hour'] = df.index.hour
    df['day_of_week'] = df.index.dayofweek
    df['month'] = df.index.month
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

    # Create lag features
    df['lag_1h'] = df[target_col].shift(1)
    df['lag_24h'] = df[target_col].shift(24)
    df['rolling_mean_6h'] = df[target_col].shift(1).rolling(window=6).mean()

    # Drop rows with missing values introduced by lag/rolling variables
    df.dropna(subset=['lag_1h', 'lag_24h', 'rolling_mean_6h'], inplace=True)

    # Drop text columns or label-encode categorical columns for numeric correlation/modeling
    categorical_cols = ['Weather Condition', 'Weather_Condition', 'Holiday Type', 'Holiday_Type', 'Festival Name']
    cols_to_use = [col for col in df.columns if col not in categorical_cols]

    X = df[cols_to_use].drop(columns=[target_col])
    y = df[target_col]

    # Train-test split (Chronological split for Time-Series evaluation)
    split_idx = int(len(df) * 0.8)
    X_train, X_val = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_val = y.iloc[:split_idx], y.iloc[split_idx:]

    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    print(f"Training set shape: {X_train.shape}, Validation set shape: {X_val.shape}")

    print("\n=================== MODEL TRAINING & EVALUATION ===================")
    models = {
        "Linear Regression": LinearRegression(),
        "Decision Tree": DecisionTreeRegressor(random_state=42, max_depth=8),
        "Random Forest": RandomForestRegressor(random_state=42, n_estimators=50, max_depth=10, n_jobs=-1),
        "XGBoost": XGBRegressor(random_state=42, n_estimators=50, max_depth=6, learning_rate=0.1)
    }

    results = []
    for name, model in models.items():
        model.fit(X_train_scaled, y_train)
        preds = model.predict(X_val_scaled)

        mae = mean_absolute_error(y_val, preds)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        r2 = r2_score(y_val, preds)

        results.append({"Model": name, "MAE": round(mae, 2), "RMSE": round(rmse, 2), "R2 Score": round(r2, 4)})

    results_df = pd.DataFrame(results)
    display(results_df)
    print("\nWorkflow completed successfully.")

run_comprehensive_pipeline('/content/load_data.csv')

   COMPREHENSIVE ENERGY DEMAND FORECASTING WORKFLOW

--- 1. LOAD DATA & INITIAL PREP ---

=================== RUNNING EDA ===================

[2] DATA QUALITY CHECK
----------------------------------------

Missing values:
load_MW                  0
Weather Condition        0
Holiday Type             0
Festival Name        22240
Temp                     0
Humidity                 0
dtype: int64

Missing percentage:
load_MW               0.00
Weather Condition     0.00
Holiday Type          0.00
Festival Name        92.76
Temp                  0.00
Humidity              0.00
dtype: float64

Duplicate rows: 0

[2.5] MISSING GAPS ANALYSIS
----------------------------------------
No missing value gaps found in the target column.

[3.5] OUTLIER CHECK (3-Sigma)
----------------------------------------
Mean = 4,425 MW, Std = 1,332 MW -> 6 hours beyond +/- 3 std devs

[3] DESCRIPTIVE STATISTICS
----------------------------------------


,load_MW,Temp,Humidity
count,23977.000000,23977.000000,23977.000000
mean,4424.555124,27.900963,58.382746
std,1331.642559,5.585656,18.722494
min,1483.680000,9.700000,25.000000
25%,3507.920000,24.100000,43.100000
50%,4339.620000,28.000000,55.400000
75%,5423.230000,32.000000,72.200000
max,8631.530000,43.600000,98.000000



=================== FEATURE ENGINEERING ===================
Training set shape: (19162, 9), Validation set shape: (4791, 9)

=================== MODEL TRAINING & EVALUATION ===================


,Model,MAE,RMSE,R2 Score
0,Linear Regression,143.25,194.75,0.9778
1,Decision Tree,122.69,166.56,0.9837
2,Random Forest,84.75,121.30,0.9914
3,XGBoost,76.82,107.67,0.9932



Workflow completed successfully.


In [ ]:
# This cell is deprecated. All code and pipeline steps have been merged successfully into the single cell above.